In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def _safe_div(a, b):
    return np.where(b != 0, a / b, 0.0)

def entropy_from_counts(cnts: np.ndarray) -> float:
    total = cnts.sum()
    if total <= 0:
        return 0.0
    p = cnts / total
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def hhi_from_counts(cnts: np.ndarray) -> float:
    total = cnts.sum()
    if total <= 0:
        return 0.0
    p = cnts / total
    return float((p * p).sum())

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0 or np.all(x == 0):
        return 0.0
    x = np.sort(np.abs(x))
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2 * (cumx.sum() / cumx[-1])) / n)

def as_float32(df):
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype(np.float32)
        elif pd.api.types.is_integer_dtype(df[c]) and c != 'customer_id':
            df[c] = df[c].astype(np.int32)
    return df

_time_re = re.compile(r'^\s*(\d{1,2}):(\d{2})(?::(\d{2}))?\s*$')
def parse_tr_datetime(s: str):
    if pd.isna(s):
        return 0, 0
    s = str(s).strip()
    parts = s.split()
    if len(parts) == 2 and parts[0].isdigit():
        day = int(parts[0])
        t = parts[1]
    else:
        day = 0
        t = parts[-1] if parts else "00:00:00"
    m = _time_re.match(t)
    if not m:
        return int(max(day, 0)), 0
    hh = int(m.group(1))
    mm = int(m.group(2))
    ss = int(m.group(3) or 0)
    hh = min(max(hh, 0), 23)
    mm = min(max(mm, 0), 59)
    ss = min(max(ss, 0), 59)
    secs = hh * 3600 + mm * 60 + ss
    return int(max(day, 0)), int(secs)

class FeatureFactoryLATTE:
    def __init__(
        self,
        topk_mcc: int = 300,
        topk_trtype: int = 60,
        topk_term: int = 60,
        tfidf_max_features_mcc: int = 800,
        tfidf_max_features_tr: int = 300,
        fixed_origin_day: int | None = None,
    ):
        self.topk_mcc = topk_mcc
        self.topk_trtype = topk_trtype
        self.topk_term = topk_term
        self.tfidf_max_features_mcc = tfidf_max_features_mcc
        self.tfidf_max_features_tr = tfidf_max_features_tr
        self.fixed_origin_day = fixed_origin_day

        self.origin_day_ = None
        self.top_mcc_ = None
        self.top_trtype_ = None
        self.top_term_ = None
        self.tfidf_mcc_ = None
        self.tfidf_tr_ = None

    # --- internal ---
    def _preprocess(self, df: pd.DataFrame, is_fit: bool):
        need = {'customer_id','tr_datetime','mcc_code','tr_type','amount','term_id'}
        miss = need - set(df.columns)
        if miss:
            raise KeyError(f"Нет колонок: {miss}")

        d = df[['customer_id','tr_datetime','mcc_code','tr_type','amount','term_id']].copy()

        d['customer_id'] = pd.to_numeric(d['customer_id'], errors='coerce').fillna(-1).astype(np.int32)
        d['mcc_code']    = pd.to_numeric(d['mcc_code'], errors='coerce').fillna(-1).astype(np.int32)
        d['tr_type']     = pd.to_numeric(d['tr_type'], errors='coerce').fillna(-1).astype(np.int32)
        d['term_id']     = pd.to_numeric(d['term_id'], errors='coerce').fillna(-1).astype(np.int32)
        d['amount']      = pd.to_numeric(d['amount'], errors='coerce').fillna(0.0).astype(np.float32)

        days, secs = zip(*d['tr_datetime'].map(parse_tr_datetime))
        d['day'] = np.asarray(days, dtype=np.int32)
        d['sec'] = np.asarray(secs, dtype=np.int32)

        if is_fit:
            self.origin_day_ = int(self.fixed_origin_day) if self.fixed_origin_day is not None else int(d['day'].min())

        d['trans_day'] = (d['day'] - self.origin_day_).astype(np.int32)
        d = d.sort_values(['customer_id','trans_day','sec']).reset_index(drop=True)

        d['pos_amount'] = d['amount'].clip(lower=0)
        d['neg_amount'] = (-d['amount'].clip(upper=0))

        return d

    def fit(self, df: pd.DataFrame):
        d = self._preprocess(df, is_fit=True)

        self.top_mcc_     = d['mcc_code'].value_counts().head(self.topk_mcc).index.tolist()
        self.top_trtype_  = d['tr_type'].value_counts().head(self.topk_trtype).index.tolist()
        self.top_term_    = d['term_id'].value_counts().head(self.topk_term).index.tolist()

        seq_mcc = (d.groupby('customer_id')['mcc_code']
                     .apply(lambda s: ' '.join(s.astype(str))))
        self.tfidf_mcc_ = TfidfVectorizer(
            max_features=self.tfidf_max_features_mcc,
            dtype=np.float32,
            token_pattern=r'(?u)\b\w+\b'
        ).fit(seq_mcc.values)

        seq_tr = (d.groupby('customer_id')['tr_type']
                    .apply(lambda s: ' '.join(s.astype(str))))
        self.tfidf_tr_ = TfidfVectorizer(
            max_features=self.tfidf_max_features_tr,
            dtype=np.float32,
            token_pattern=r'(?u)\b\w+\b'
        ).fit(seq_tr.values)

        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if any(x is None for x in [self.origin_day_, self.top_mcc_, self.top_trtype_, self.top_term_, self.tfidf_mcc_, self.tfidf_tr_]):
            raise RuntimeError("Сначала вызови .fit() на train.")

        d = self._preprocess(df, is_fit=False)

        agg = d.groupby('customer_id').agg(
            tx_cnt=('amount','size'),
            sum_amount=('amount','sum'),
            mean_amount=('amount','mean'),
            std_amount=('amount','std'),
            min_amount=('amount','min'),
            max_amount=('amount','max'),
            q25_amount=('amount', lambda x: np.percentile(x, 25)),
            q50_amount=('amount', lambda x: np.percentile(x, 50)),
            q75_amount=('amount', lambda x: np.percentile(x, 75)),
            sum_income=('pos_amount','sum'),
            sum_expense=('neg_amount','sum'),
            cnt_income=('pos_amount', lambda x: (x > 0).sum()),
            cnt_expense=('neg_amount', lambda x: (x > 0).sum()),
            first_day=('trans_day','min'),
            last_day=('trans_day','max'),
            active_days=('trans_day','nunique'),
            uniq_mcc=('mcc_code','nunique'),
            uniq_trtype=('tr_type','nunique'),
            uniq_term=('term_id','nunique'),
        ).reset_index()

        agg['tx_cnt'] = agg['tx_cnt'].astype(np.int32)
        agg['period'] = (agg['last_day'] - agg['first_day'] + 1).astype(np.int32)
        agg['tx_per_day'] = _safe_div(agg['tx_cnt'].astype(np.float32), agg['period'].astype(np.float32))
        agg['days_ratio'] = _safe_div(agg['active_days'].astype(np.float32), agg['period'].astype(np.float32))
        agg['income_share']  = _safe_div(agg['sum_income'], agg['sum_amount'].abs() + 1e-9)
        agg['expense_share'] = _safe_div(agg['sum_expense'], agg['sum_amount'].abs() + 1e-9)

        day_agg = (d.groupby(['customer_id','trans_day'])
                     .agg(day_sum=('amount','sum'),
                          day_cnt=('amount','size'),
                          day_pos=('pos_amount','sum'),
                          day_neg=('neg_amount','sum'))
                     .reset_index())
        daily = day_agg.groupby('customer_id').agg(
            day_sum_mean=('day_sum','mean'),
            day_sum_std =('day_sum','std'),
            day_cnt_mean=('day_cnt','mean'),
            day_cnt_std =('day_cnt','std'),
            day_sum_q90 =('day_sum', lambda x: np.percentile(x, 90)),
            day_pos_mean=('day_pos','mean'),
            day_neg_mean=('day_neg','mean'),
        ).reset_index()

        def _gap_stats(s):
            v = np.sort(s.values)
            if v.size <= 1:
                return pd.Series(dict(gap_mean=0.0, gap_median=0.0))
            gaps = np.diff(v)
            return pd.Series(dict(gap_mean=float(gaps.mean()), gap_median=float(np.median(gaps))))
        gaps = d.groupby('customer_id')['trans_day'].apply(_gap_stats).reset_index()
        gaps = gaps.pivot(index='customer_id', columns='level_1', values='trans_day').reset_index()

        def _dist_stats(df_g, col, prefix):
            cnt = df_g.groupby(['customer_id', col]).size().rename('cnt').reset_index()
            g = cnt.groupby('customer_id')['cnt']
            stats = pd.DataFrame({
                f'{prefix}_entropy': g.apply(lambda s: entropy_from_counts(s.to_numpy())),
                f'{prefix}_hhi':     g.apply(lambda s: hhi_from_counts(s.to_numpy())),
                f'{prefix}_top1_share': g.apply(lambda s: float(s.max() / s.sum()) if s.sum() > 0 else 0.0),
            }).reset_index()
            return stats

        mcc_stats  = _dist_stats(d, 'mcc_code', 'mcc')
        trt_stats  = _dist_stats(d, 'tr_type', 'trtype')
        term_stats = _dist_stats(d, 'term_id', 'term')

        def _top_pivots_fixed(df_g, col, top_vals, prefix, val_col='amount'):
            if len(top_vals) == 0:
                return (pd.DataFrame({'customer_id': df_g['customer_id'].unique()}),
                        pd.DataFrame({'customer_id': df_g['customer_id'].unique()}),
                        pd.DataFrame({'customer_id': df_g['customer_id'].unique()}))
            mask = df_g[col].isin(top_vals)
            p_cnt = df_g[mask].groupby(['customer_id', col]).size().unstack(fill_value=0)
            p_cnt = p_cnt.reindex(columns=top_vals, fill_value=0).add_prefix(f'{prefix}cnt_').reset_index()

            p_sum = df_g[mask].groupby(['customer_id', col])[val_col].sum().unstack(fill_value=0.0)
            p_sum = p_sum.reindex(columns=top_vals, fill_value=0.0).add_prefix(f'{prefix}sum_').reset_index()

            p_mean = df_g[mask].groupby(['customer_id', col])[val_col].mean().unstack()
            p_mean = p_mean.reindex(columns=top_vals).fillna(0.0).add_prefix(f'{prefix}mean_').reset_index()
            return p_cnt, p_sum, p_mean

        mcc_cnt, mcc_sum, mcc_mean   = _top_pivots_fixed(d, 'mcc_code', self.top_mcc_,   'mcc_')
        trt_cnt, trt_sum, trt_mean   = _top_pivots_fixed(d, 'tr_type',  self.top_trtype_,'trt_')
        term_cnt, term_sum, term_mean= _top_pivots_fixed(d, 'term_id',  self.top_term_,  'term_')

        seq_mcc = (d.groupby('customer_id')['mcc_code'].apply(lambda s: ' '.join(s.astype(str))))
        X_mcc = self.tfidf_mcc_.transform(seq_mcc.values)
        mcc_cols = [f'tfidf_mcc_{t}' for t in self.tfidf_mcc_.get_feature_names_out()]
        mcc_df = pd.DataFrame(X_mcc.toarray(), columns=mcc_cols)
        mcc_df.insert(0, 'customer_id', seq_mcc.index.values.astype(np.int32))

        seq_tr = (d.groupby('customer_id')['tr_type'].apply(lambda s: ' '.join(s.astype(str))))
        X_tr = self.tfidf_tr_.transform(seq_tr.values)
        tr_cols = [f'tfidf_tr_{t}' for t in self.tfidf_tr_.get_feature_names_out()]
        tr_df = pd.DataFrame(X_tr.toarray(), columns=tr_cols)
        tr_df.insert(0, 'customer_id', seq_tr.index.values.astype(np.int32))

        gini_series = (
            d.groupby('customer_id')['amount'].apply(lambda s: gini(s.to_numpy())).astype(np.float32)
        ).rename('amount_gini').reset_index()

        feats = (agg
                 .merge(daily, on='customer_id', how='left')
                 .merge(gaps, on='customer_id', how='left')
                 .merge(mcc_stats, on='customer_id', how='left')
                 .merge(trt_stats, on='customer_id', how='left')
                 .merge(term_stats, on='customer_id', how='left')
                 .merge(mcc_cnt, on='customer_id', how='left')
                 .merge(mcc_sum, on='customer_id', how='left')
                 .merge(mcc_mean, on='customer_id', how='left')
                 .merge(trt_cnt, on='customer_id', how='left')
                 .merge(trt_sum, on='customer_id', how='left')
                 .merge(trt_mean, on='customer_id', how='left')
                 .merge(term_cnt, on='customer_id', how='left')
                 .merge(term_sum, on='customer_id', how='left')
                 .merge(term_mean, on='customer_id', how='left')
                 .merge(mcc_df, on='customer_id', how='left')
                 .merge(tr_df, on='customer_id', how='left')
                 .merge(gini_series, on='customer_id', how='left')
                 )

        feats = feats.replace([np.inf, -np.inf], 0.0).fillna(0.0)
        feats = as_float32(feats)
        return feats

In [2]:
seed_everywhere = 556

np.random.seed(seed_everywhere)

In [3]:
COL_ID = 'customer_id'
COL_TARGET = 'gender'

train = pd.read_csv('/home/jovyan/zoloev-madvillainy/LATTE/data/gender/train.csv')
test = pd.read_csv('/home/jovyan/zoloev-madvillainy/LATTE/data/gender/test.csv')

train_ids = pd.read_csv('/home/jovyan/zoloev-madvillainy/LATTE/data/gender/meta/gender_train.csv')
test_ids = pd.read_csv('/home/jovyan/zoloev-madvillainy/LATTE/data/gender/meta/test_ids.csv')

train_transactions = train[
    train["customer_id"].isin(set(train_ids["customer_id"]))
].reset_index(drop=True)

test_transactions = test[
    test["customer_id"].isin(set(test_ids["customer_id"]))
].reset_index(drop=True)

TurboTransformer = FeatureFactoryLATTE(
    topk_mcc=300, topk_trtype=60, topk_term=60,
    tfidf_max_features_mcc=800, tfidf_max_features_tr=300,
    fixed_origin_day=None,
).fit(train_transactions)

train_turbo_stats = TurboTransformer.transform(train_transactions)
test_turbo_stats = TurboTransformer.transform(test_transactions)

train_test_turbo_stats = pd.concat(
    (train_turbo_stats, test_turbo_stats)
)

In [4]:
X_train = train_test_turbo_stats.merge(train_turbo_stats[COL_ID], on=COL_ID)
X_train = X_train.merge(train_ids, on=COL_ID)

X_train['random_feature'] = np.random.normal(size=7560)

In [5]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    logging_level='Silent',
    allow_writing_files=False,
    random_state=seed_everywhere
)

model.fit(
    X_train.drop(columns=[COL_ID, COL_TARGET]),
    X_train[COL_TARGET]
)

In [6]:
importances = pd.Series(
    model.feature_importances_, 
    index=X_train.drop(columns=[COL_ID, COL_TARGET]).columns
)
# top_features = importances.sort_values(ascending=False).head(128)

In [7]:
top_features = importances[importances > importances.sort_values(ascending=False)['random_feature']].sort_values(ascending=True)

In [8]:
train_test_turbo_stats = train_test_turbo_stats[[COL_ID] + list(top_features.index)]
train_test_turbo_stats.attrs = {}
train_test_turbo_stats.to_parquet(
    '/home/jovyan/zoloev-madvillainy/LATTE/data/gender/statistics/turbo_stats_gender_selected_seed_556.parquet',
)

In [9]:
# train_test_turbo_stats = pd.read_parquet('/home/jovyan/zoloev-madvillainy/LATTE/data/gender/statistics/turbo_stats_gender_selected.parquet')

---

In [18]:
# def feature_to_text(features):
#     # text = [f"{col}: {features[col]}" for col in features.index[1:]]
#     text = [f"{features[col]}" for col in features.index[1:]]
#     return "\n".join(text)

# import numpy as np

def feature_to_text(features, seed=None, include_first=False, method="random"):
    idx = list(features.index)
    if not include_first:
        cols = idx[1:]
    else:
        cols = idx[:]

    n = len(cols)
    if n == 0:
        return ""

    vals = [features[c] for c in cols]

    if n == 1:
        names = cols[:]
    elif method == "shift":
        names = cols[1:] + cols[:1]
    else:
        rng = np.random.default_rng(seed)
        if n == 2:
            names = cols[::-1]
        else:
            while True:
                p = rng.permutation(n)
                if np.all(p != np.arange(n)):
                    break
            names = [cols[i] for i in p]

    lines = [f"{name}: {val}" for name, val in zip(names, vals)]
    return "\n".join(lines)

In [19]:
train_test_turbo_stats

,customer_id,mcc_mean_5999,tfidf_tr_4051,mcc_mean_5661,tfidf_mcc_6051,trt_mean_1110,trt_mean_2010,min_amount,std_amount,mcc_mean_6011,...,tfidf_mcc_5621,tfidf_mcc_5651,mcc_mean_5977,tfidf_mcc_5541,mcc_cnt_5533,tfidf_mcc_5912,tfidf_mcc_5533,mcc_sum_5541,tfidf_mcc_5661,tfidf_mcc_5977
0,22899,0.000000,0.048148,0.000000,0.000000,-10555.670898,-56802.953125,-8.686079e+05,127851.640625,13905.199219,...,0.000000,0.000000,0.000000,0.113179,0,0.010935,0.000000,-54805.519531,0.000000,0.000000
1,28753,-217040.390625,0.000000,-171700.250000,0.000000,-66132.492188,-418676.125000,-3.244540e+06,785586.187500,-199682.328125,...,0.000000,0.000000,-62997.941406,0.074158,0,0.066875,0.000000,-182942.437500,0.026116,0.013602
2,42096,-10388.257812,0.000000,-18761.816406,0.018545,-18331.556641,-33981.683594,-3.377857e+05,63639.187500,-27129.667969,...,0.000000,0.010306,0.000000,0.000000,0,0.134019,0.000000,0.000000,0.035743,0.000000
3,49793,-5951.680176,0.000000,0.000000,0.000000,-4223.717285,-331529.250000,-6.737748e+05,161784.406250,-278370.312500,...,0.000000,0.000000,0.000000,0.000000,1,0.003248,0.005712,0.000000,0.000000,0.000000
4,50940,0.000000,0.000000,0.000000,0.000000,-4341.691895,-18990.466797,-1.100499e+05,18341.187500,-18990.466797,...,0.000000,0.000000,0.000000,0.000000,0,0.028823,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8395,99452850,0.000000,0.000000,-86832.718750,0.000000,-33341.226562,-199613.343750,-3.144282e+06,473367.812500,-180489.093750,...,0.018957,0.030940,-31838.099609,0.000000,0,0.112850,0.000000,0.000000,0.026825,0.034928
8396,99486577,0.000000,0.000000,0.000000,0.000000,-16900.820312,0.000000,-5.839381e+05,77560.859375,-70039.304688,...,0.017840,0.014559,-6785.475098,0.000000,0,0.110819,0.000000,0.000000,0.000000,0.052593
8397,99673663,-5237.476074,0.000000,-22971.228516,0.000000,-19456.369141,-23012.500000,-2.245916e+05,19252.642578,-21975.664062,...,0.000000,0.000000,0.000000,0.000000,0,0.071595,0.000000,0.000000,0.097857,0.000000
8398,99900908,0.000000,0.134599,0.000000,0.000000,-6050.495117,-124921.484375,-3.818057e+05,104280.117188,-117215.414062,...,0.000000,0.000000,0.000000,0.000000,0,0.271214,0.000000,0.000000,0.000000,0.000000


In [20]:
train_test_turbo_stats = train_test_turbo_stats.reset_index(drop=True)

stats_text = pd.DataFrame({
    'customer_id': train_test_turbo_stats['customer_id'],
    'text': train_test_turbo_stats.apply(feature_to_text, axis=1)
})

stats_text['text'] = '''You are given tabular client features extracted from bank transaction history.
Each feature name represents a specific behavioral or financial metric.
Below is a mapping from feature names to their meaning:\n''' + stats_text['text']

print(stats_text['text'][0])

You are given tabular client features extracted from bank transaction history.
Each feature name represents a specific behavioral or financial metric.
Below is a mapping from feature names to their meaning:
mcc_mean_4829: 0.0
mcc_cnt_6011: 0.04814814776182175
mcc_mean_5912: 0.0
tfidf_mcc_5541: 0.0
mcc_mean_5661: -10555.6708984375
day_neg_mean: -56802.953125
mcc_mean_5411: -868607.9375
mcc_sum_5541: 127851.640625
tfidf_mcc_5983: 13905.19921875
tfidf_mcc_5511: 0.0
mean_amount: 1362709.5
trt_sum_2010: 0.0
tfidf_mcc_6051: 0.0
tfidf_mcc_5631: 0.0
tfidf_tr_1030: -285003.125
tfidf_mcc_6010: 0.0
day_cnt_std: -2691.4384765625
tfidf_mcc_5977: 0.0
mcc_sum_5621: 0.0
mcc_sum_5921: 2.1666667461395264
tfidf_mcc_5941: 1.0
mcc_cnt_5533: 0.0
tfidf_mcc_5921: 0.0
term_sum_-1: 47.0
mcc_cnt_5732: 0.20012372732162476
mcc_mean_6010: 0.2368421107530594
tfidf_mcc_5661: -3737.739990234375
day_sum_std: 0.0
trt_cnt_2330: -37422.578125
trtype_entropy: 58601.63671875
sum_expense: 0.0
min_amount: 0.0
max_amount: 0.0


In [21]:
stats_text.to_csv(
    '/home/jovyan/zoloev-madvillainy/LATTE/data/gender/statistics/turbo_stats_gender_selected_random.csv',
    index=False
)

In [ ]:
turbo_stats_gender_selected_base
turbo_stats_gender_selected_value
turbo_stats_gender_selected_random